In [1]:
#preprocess delle zones simile a come viene fatto in ghosh kuiper mahes maragno 2022
import pandas as pd

In [2]:
df = pd.read_csv('../data_flat/training_dataset.csv')

## prendo solo routes con score high

In [3]:
df = df[df['route_score']=='High']

In [4]:
# a questo livello non si esclude che una zona appaia più volte: potrebbe a priori apparire più volte intervallata da altre

# rimuovo la station (zone_id mancante)
df_zones = df[df["zone_id"].notna()].copy()

all_routes = []

for route_id, route_df in df_zones.groupby("route_id"):

    route_df = route_df.sort_values("actual_sequence").copy()

    # identifica un nuovo gruppo quando cambia zona
    route_df["zone_block"] = (
        route_df["zone_id"] != route_df["zone_id"].shift()
    ).cumsum()

    route_summary = (
        route_df
        .groupby("zone_block")
        .agg(
            route_id=("route_id", "first"),
            zone_id=("zone_id", "first"),
            num_stops_in_zone=("stop_id", "count"),
            first_actual_sequence=("actual_sequence", "min"),
            last_actual_sequence=("actual_sequence", "max"),
            min_lat=("lat", "min"),
            max_lat=("lat", "max"),
            min_lng=("lng", "min"),
            max_lng=("lng", "max"),
            center_lat=("lat", "mean"),
            center_lon=("lng", "mean"),
        )
        .reset_index(drop=True)
    )

    route_summary["zone_sequence"] = range(
        1,
        len(route_summary) + 1
    )

    all_routes.append(route_summary)

zone_sequences = pd.concat(all_routes, ignore_index=True)

zone_sequences.head()

,route_id,zone_id,num_stops_in_zone,first_actual_sequence,last_actual_sequence,min_lat,max_lat,min_lng,max_lng,center_lat,center_lon,zone_sequence
0,RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77,A-2.2A,3,1,3,34.088467,34.088717,-118.286484,-118.284521,34.088631,-118.285281,1
1,RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77,A-2.1A,2,4,5,34.089727,34.090901,-118.285530,-118.284257,34.090314,-118.284893,2
2,RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77,A-1.1A,2,6,7,34.091721,34.092285,-118.284539,-118.283236,34.092003,-118.283887,3
3,RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77,A-1.2A,6,8,13,34.092890,34.094506,-118.284798,-118.282396,34.093508,-118.283406,4
4,RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77,A-1.3A,1,14,14,34.095810,34.095810,-118.284534,-118.284534,34.095810,-118.284534,5


In [5]:
zone_sequences.to_csv(
    "zone_sequences.csv",
    index=False
)


In [6]:
#faccio la riduzione come nell'articolo: se una zona appare più volte per route tengo solo quella con maggior num di stop. se c'è ancora equivalenza tengo solo la prima occorrenza


zone_sequences = pd.read_csv("../data_flat/zone_sequences.csv")

# ordina in modo che la prima riga per ogni zona sia quella da mantenere
zone_sequences_reduced = (
    zone_sequences
    .sort_values(
        [
            "route_id",
            "zone_id",
            "num_stops_in_zone",
            "zone_sequence"
        ],
        ascending=[
            True,
            True,
            False,   # più stop prima
            True     # in caso di parità prima occorrenza
        ]
    )
    .drop_duplicates(
        subset=["route_id", "zone_id"],
        keep="first"
    )
)

# riordino secondo la sequenza scelta
zone_sequences_reduced = (
    zone_sequences_reduced
    .sort_values(
        ["route_id", "zone_sequence"]
    )
    .reset_index(drop=True)
)

# nuova sequenza delle zone dopo la riduzione
zone_sequences_reduced["zone_sequence_reduced"] = (
    zone_sequences_reduced
    .groupby("route_id")
    .cumcount()
    + 1
)

In [7]:
zone_sequences_reduced

,route_id,zone_id,num_stops_in_zone,first_actual_sequence,last_actual_sequence,min_lat,max_lat,min_lng,max_lng,center_lat,center_lon,zone_sequence,zone_sequence_reduced
0,RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77,A-2.2A,3,1,3,34.088467,34.088717,-118.286484,-118.284521,34.088631,-118.285281,1,1
1,RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77,A-2.1A,2,4,5,34.089727,34.090901,-118.285530,-118.284257,34.090314,-118.284893,2,2
2,RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77,A-1.1A,2,6,7,34.091721,34.092285,-118.284539,-118.283236,34.092003,-118.283888,3,3
3,RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77,A-1.2A,6,8,13,34.092890,34.094506,-118.284798,-118.282396,34.093508,-118.283406,4,4
4,RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77,A-1.3A,6,16,21,34.091500,34.095431,-118.286891,-118.285547,34.093342,-118.286300,7,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
54304,RouteID_fff2548c-1069-4340-8e50-08420efe435d,J-7.3J,5,52,56,34.067471,34.068619,-118.357535,-118.355416,34.068165,-118.356651,17,14
54305,RouteID_fff2548c-1069-4340-8e50-08420efe435d,J-7.1J,6,58,63,34.065114,34.066381,-118.358478,-118.356964,34.065707,-118.357532,19,15
54306,RouteID_fff2548c-1069-4340-8e50-08420efe435d,J-8.3J,2,64,65,34.065631,34.065984,-118.359609,-118.359218,34.065808,-118.359414,20,16
54307,RouteID_fff2548c-1069-4340-8e50-08420efe435d,J-8.2J,4,66,69,34.066432,34.067341,-118.360427,-118.358482,34.066883,-118.359287,21,17


In [8]:
zone_sequences_reduced = zone_sequences_reduced[[
    "route_id",
    "zone_id",
    "min_lat",
    "max_lat",
    "min_lng",
    "max_lng",
    "center_lat",
    "center_lon",
"zone_sequence_reduced"]].copy()

In [9]:
zone_sequences_reduced[zone_sequences_reduced['route_id']=='RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77']

,route_id,zone_id,min_lat,max_lat,min_lng,max_lng,center_lat,center_lon,zone_sequence_reduced
0,RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77,A-2.2A,34.088467,34.088717,-118.286484,-118.284521,34.088631,-118.285281,1
1,RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77,A-2.1A,34.089727,34.090901,-118.285530,-118.284257,34.090314,-118.284893,2
2,RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77,A-1.1A,34.091721,34.092285,-118.284539,-118.283236,34.092003,-118.283888,3
3,RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77,A-1.2A,34.092890,34.094506,-118.284798,-118.282396,34.093508,-118.283406,4
4,RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77,A-1.3A,34.091500,34.095431,-118.286891,-118.285547,34.093342,-118.286300,5
5,RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77,A-1.3B,34.092428,34.095705,-118.289226,-118.287525,34.094591,-118.288184,6
6,RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77,A-1.1B,34.091648,34.094510,-118.291761,-118.288950,34.093092,-118.290820,7
7,RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77,A-1.1C,34.090390,34.094216,-118.293884,-118.292792,34.092132,-118.293313,8
8,RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77,A-1.2C,34.096132,34.097468,-118.292869,-118.290254,34.096974,-118.291626,9
9,RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77,A-1.3D,34.098157,34.098157,-118.289765,-118.289765,34.098157,-118.289765,10


In [10]:
#creo un dataset per training con zone ridotte come nel paper per toute con score HIGH
zone_sequences_reduced.to_csv(
    "zone_sequences_reduced.csv",
    index=False
)

# plot del percorso per zone 'reduced'

In [1]:
import pandas as pd
import altair as alt

zone_sequences_reduced = pd.read_csv(
    "../data_flat/zone_sequences_reduced.csv")
route_id = "RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77"

route = (
    zone_sequences_reduced[
        zone_sequences_reduced["route_id"] == route_id
    ]
    .sort_values("zone_sequence_reduced")
    .copy()
)

# centroidi dei box

route["lng"] = route["center_lon"]

route["lat"] = route["center_lat"]



#bounding box

boxes = []

for _, r in route.iterrows():

    vertices = [
        (r["min_lng"], r["min_lat"]),
        (r["max_lng"], r["min_lat"]),
        (r["max_lng"], r["max_lat"]),
        (r["min_lng"], r["max_lat"]),
        (r["min_lng"], r["min_lat"])
    ]

    for order, (lng, lat) in enumerate(vertices):

        boxes.append({
            "route_id": r["route_id"],
            "zone_id": r["zone_id"],
            "zone_sequence_reduced": r["zone_sequence_reduced"],
            "lng": lng,
            "lat": lat,
            "order": order
        })

boxes_df = pd.DataFrame(boxes)

#rettangoli

rectangles = (
    alt.Chart(boxes_df)
    .mark_line(strokeWidth=2)
    .encode(
        x=alt.X("lng:O"),
        y=alt.Y("lat:O"),
        color=alt.Color("zone_id:N"),
        detail="zone_sequence_reduced:N",
        order="order:Q",
        tooltip=[
            "zone_id:N",
            "zone_sequence_reduced:Q"
        ]
    )
)

#percorso secondo zone seq reduced

path = (
    alt.Chart(route)
    .mark_line(
        color="black",
        strokeWidth=3
    )
    .encode(
        x=alt.X("lng:O"),
        y=alt.Y("lat:O"),
        order="zone_sequence_reduced:Q"
    )
)

#centro zona come media

points = (
    alt.Chart(route)
    .mark_circle(size=150)
    .encode(
        x=alt.X("lng:O"),
        y=alt.Y("lat:O"),
        color=alt.Color("zone_id:N", legend=None),
        tooltip=[
            "zone_id:N",
            "zone_sequence_reduced:Q"
        ]
    )
)

#labels

labels = (
    alt.Chart(route)
    .mark_text(
        dx=8,
        dy=-8,
        fontSize=10,
        color="black"
    )
    .encode(
        x=alt.X("lng:O"),
        y=alt.Y("lat:O"),
        text="zone_sequence_reduced:Q"
    )
)

chart = (
    alt.layer(
        rectangles,
        path,
        points,
        labels
    )
    .encode(
       x=alt.X(
    "lng:O",
    title="Longitude",
    axis=alt.Axis(labels=False)
),
y=alt.Y(
    "lat:O",
    title="Latitude",
    axis=alt.Axis(labels=False)
)
    )
    .properties(
        width=900,
        height=700,
        title=f"Zone sequence: {route_id}"
    )
)

chart

alt.LayerChart(...)